# Baselines B0 and B1

Simple models that set the score our image models must beat.

| ID | Model | Input |
| --- | --- | --- |
| B0 | Training-set mean for every series | Nothing |
| B1 | Logistic regression / LightGBM | Age, sex, modality |

**Evaluation (same for every model, also B2 and the final model):**
- The shared 5-fold split `splits/folds.csv` (`src.utils.load_folds`).
- For each fold, the model is trained on the other 4 folds and predicts this fold (`src.utils.run_cv`).
  The result is one out-of-fold (OOF) prediction for every series.
- Each fold is scored with the competition metric on its own, then the 5 scores are averaged (`src.utils.cv_report`).
- OOF predictions are saved to `results/oof_<model>.csv` for the final comparison.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.utils import LABEL_COLS, cv_report, load_folds, run_cv, save_oof

train = pd.read_csv(ROOT / "data" / "raw" / "train.csv")
folds = load_folds()
results = {}  # model name -> summary scores


def show(name, report):
    """Print the scores of one model and keep its summary for the final table."""
    results[name] = report["summary"]
    print(report["summary"].round(4).to_string())
    print("\nScore per fold:", report["per_fold"].round(4).tolist())
    print("\nScore per modality:")
    print(report["per_modality"].round(4).to_string())

## B0 — training-set mean

For every label, B0 predicts the share of positives in the training folds, the same number for every series.
AUC only looks at the *order* of the predictions, and here every series gets the same value,
so every AUC is exactly 0.5. B0 is a sanity check of the evaluation code: if it scored above 0.5,
labels from the validation fold would be leaking into the predictions.

In [2]:
def b0_fit_predict(train_part, valid_part):
    means = train_part[LABEL_COLS].mean().to_numpy()
    return np.tile(means, (len(valid_part), 1))


oof_b0 = run_cv(train, b0_fit_predict, folds)
show("B0", cv_report(train, oof_b0))
save_oof(oof_b0, "b0")
oof_b0.head(3)

cv_score             0.5
cv_std               0.0
present_auc          0.5
mean_location_auc    0.5

Score per fold: [0.5, 0.5, 0.5, 0.5, 0.5]

Score per modality:
CTA           0.5
MRA           0.5
MRI T1post    0.5
MRI T2        0.5


,SeriesInstanceUID,fold,Left Infraclinoid Internal Carotid Artery,Right Infraclinoid Internal Carotid Artery,Left Supraclinoid Internal Carotid Artery,Right Supraclinoid Internal Carotid Artery,Left Middle Cerebral Artery,Right Middle Cerebral Artery,Anterior Communicating Artery,Left Anterior Cerebral Artery,Right Anterior Cerebral Artery,Left Posterior Communicating Artery,Right Posterior Communicating Artery,Basilar Tip,Other Posterior Circulation,Aneurysm Present
0,1.2.826.0.1.3680043.8.498.10004044428023505108...,1,0.018401,0.022139,0.075906,0.064117,0.049741,0.067855,0.083381,0.010063,0.012938,0.020127,0.022714,0.025014,0.026164,0.428695
1,1.2.826.0.1.3680043.8.498.10004684224894397679...,3,0.017534,0.022995,0.075596,0.064386,0.050877,0.065536,0.084794,0.010635,0.012935,0.020121,0.023570,0.025582,0.025870,0.428284
2,1.2.826.0.1.3680043.8.498.10005158603912009425...,1,0.018401,0.022139,0.075906,0.064117,0.049741,0.067855,0.083381,0.010063,0.012938,0.020127,0.022714,0.025014,0.026164,0.428695


As expected, B0 scores 0.5 on every fold, every label and every modality.
The prediction is the same for all series of a fold (about 0.43 for Aneurysm Present) and differs only slightly between folds.

## B1 — age, sex and modality

EDA 1 showed that the positive rate depends on modality (CTA 54%, MRI 25%), sex (women 47%, men 34%)
and age (29% under 40, about 46% over 50). B1 uses only these three columns, no image.

We train one model per label (14 models per fold) and try two model types:
- **B1-LR: logistic regression.** Modality and sex are one-hot encoded. The effect of age is not a straight line
  (it rises until about 50, then stays flat), so age goes through a spline: a smooth curve the model can bend.
- **B1-LGBM: LightGBM.** Gradient-boosted decision trees, which can also learn combinations
  (for example, that the sex effect is larger in CTA than in MRI T1post). Small trees and a high minimum leaf size,
  because there are only 3 features and some labels have few positives.

We tried only a few settings (linear vs spline age; 100/200/400 trees); they changed the score by less than 0.01.

In [3]:
import lightgbm as lgb
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, SplineTransformer

FEATURES = ["PatientAge", "PatientSex", "Modality"]


def make_lr():
    features = ColumnTransformer([
        ("age", SplineTransformer(n_knots=5, degree=3), ["PatientAge"]),
        ("categories", OneHotEncoder(drop="if_binary"), ["PatientSex", "Modality"]),
    ])
    return make_pipeline(features, LogisticRegression(max_iter=1000))


def b1_lr_fit_predict(train_part, valid_part):
    preds = []
    for col in LABEL_COLS:
        model = make_lr().fit(train_part[FEATURES], train_part[col])
        preds.append(model.predict_proba(valid_part[FEATURES])[:, 1])
    return np.column_stack(preds)


oof_b1_lr = run_cv(train, b1_lr_fit_predict, folds)
show("B1-LR", cv_report(train, oof_b1_lr))
_ = save_oof(oof_b1_lr, "b1_lr")

cv_score             0.6592
cv_std               0.0070
present_auc          0.6749
mean_location_auc    0.6436

Score per fold: [0.6572, 0.6491, 0.66, 0.6616, 0.6682]

Score per modality:
CTA           0.5833
MRA           0.6020
MRI T1post    0.5954
MRI T2        0.5964


In [4]:
MODALITIES = sorted(train["Modality"].unique())
LGBM_PARAMS = dict(n_estimators=200, learning_rate=0.03, num_leaves=7, min_child_samples=50,
                   subsample=0.8, subsample_freq=1, random_state=0, verbose=-1)


def lgbm_features(df):
    return pd.DataFrame({
        "age": df["PatientAge"],
        "male": (df["PatientSex"] == "Male").astype(int),
        "modality": pd.Categorical(df["Modality"], categories=MODALITIES),
    })


def b1_lgbm_fit_predict(train_part, valid_part):
    X_train, X_valid = lgbm_features(train_part), lgbm_features(valid_part)
    preds = []
    for col in LABEL_COLS:
        model = lgb.LGBMClassifier(**LGBM_PARAMS).fit(X_train, train_part[col])
        preds.append(model.predict_proba(X_valid)[:, 1])
    return np.column_stack(preds)


oof_b1_lgbm = run_cv(train, b1_lgbm_fit_predict, folds)
show("B1-LGBM", cv_report(train, oof_b1_lgbm))
_ = save_oof(oof_b1_lgbm, "b1_lgbm")

cv_score             0.6656
cv_std               0.0100
present_auc          0.6760
mean_location_auc    0.6552

Score per fold: [0.6624, 0.6534, 0.6602, 0.6766, 0.6753]

Score per modality:
CTA           0.5902
MRA           0.6192
MRI T1post    0.6498
MRI T2        0.6676


### What did B1 learn?

The logistic regression coefficients for *Aneurysm Present*, trained on all data.
A positive coefficient raises the predicted probability. Age is a spline, so its effect is shown as a curve instead.

In [5]:
lr = make_lr().fit(train[FEATURES], train["Aneurysm Present"])
names = lr[0].get_feature_names_out()
coef = pd.Series(lr[-1].coef_[0], index=names)
print("Sex coefficient (reference: female) and one coefficient per modality (compare them with each other):")
print(coef[[n for n in names if n.startswith("categories")]].round(2).to_string())

ages = pd.DataFrame({"PatientAge": np.arange(20, 90, 10), "PatientSex": "Female", "Modality": "CTA"})
print("\nPredicted probability for a woman with CTA, by age:")
print(pd.Series(lr.predict_proba(ages)[:, 1].round(2), index=ages["PatientAge"]).to_string())

Sex coefficient (reference: female) and one coefficient per modality (compare them with each other):
categories__PatientSex_Male       -0.64
categories__Modality_CTA           0.60
categories__Modality_MRA           0.18
categories__Modality_MRI T1post   -0.67
categories__Modality_MRI T2       -0.64

Predicted probability for a woman with CTA, by age:
PatientAge
20    0.22
30    0.39
40    0.57
50    0.61
60    0.62
70    0.64
80    0.61


### Score per modality

Inside one modality, the modality feature is the same for every series, so B1 can only use age and sex.

In [6]:
per_modality = pd.DataFrame({
    name: cv_report(train, oof)["per_modality"]
    for name, oof in [("B0", oof_b0), ("B1-LR", oof_b1_lr), ("B1-LGBM", oof_b1_lgbm)]
})
per_modality.round(3)

,B0,B1-LR,B1-LGBM
CTA,0.5,0.583,0.590
MRA,0.5,0.602,0.619
MRI T1post,0.5,0.595,0.650
MRI T2,0.5,0.596,0.668


## Results

In [7]:
pd.DataFrame(results).T.round(4)

,cv_score,cv_std,present_auc,mean_location_auc
B0,0.5000,0.000,0.5000,0.5000
B1-LR,0.6592,0.007,0.6749,0.6436
B1-LGBM,0.6656,0.010,0.6760,0.6552


## Findings

- **B0 = 0.500** on every fold, label and modality: the evaluation code does not leak labels.
- **B1 ≈ 0.66** without looking at any image (LR 0.659, LightGBM 0.666, fold std about 0.01).
  All three features matter, in the same direction as in EDA 1: CTA and MRA above MRI, women above men,
  and a much lower probability under 40.
- The location AUCs (about 0.65) are also above 0.5. B1 cannot know *where* an aneurysm is; it mostly ranks
  series with any aneurysm higher, and that also ranks the location labels.
- Within one modality the score is lower: 0.58–0.59 for CTA and 0.60–0.67 for the others, where only age and sex
  are left. LightGBM is better here than LR, because it can learn that age and sex matter differently per modality.
- **An image model has to clearly beat about 0.66** to show that it uses the images, and not only
  who the patient is and which scanner was used.